# Ultralytics video inference on `.mp4`

Run object detection on a video and save an annotated output video with bounding boxes.

Update the paths in the next cell, then run the notebook top to bottom.


In [1]:
import sys
sys.path.insert(0, "/cluster/home/henrban/aquaculture-perception")
import ultralytics
print(ultralytics.__file__)

import sys, os
print("python:", sys.executable)
print("cwd:", os.getcwd())

try:
    import ultralytics
    print("ultralytics imported from:", ultralytics.__file__)
except Exception as e:
    print("ultralytics import failed:", repr(e))


/cluster/home/henrban/aquaculture-perception/ultralytics/__init__.py
python: /cluster/home/henrban/aquaculture-perception/.venv/bin/python
cwd: /cluster/home/henrban/aquaculture-perception/object-detection
ultralytics imported from: /cluster/home/henrban/aquaculture-perception/ultralytics/__init__.py


In [2]:
from pathlib import Path

# Use the current working directory as the project root.
ROOT = Path.cwd()

# --- EDIT THESE PATHS ---
MODEL_PATH = ROOT / '../runs/detect/outputs/training/solaqua_fish/yolov8n_solaqua_fish_120e_fair/weights/best.pt'
VIDEO_PATH = ROOT / '../data-processing/vision/SOLAQUA/raw_processed/mp4s/vision_raw_2024-08-20_17-14-36.mp4'
OUTPUT_ROOT = ROOT / '../runs/video_demos/predictions/solaqua_fish/'
RUN_NAME = 'VIDEO_yolov8n_solaqua_fish_120e_fair'

# Optional inference settings
IMGSZ = 640
CONF = 0.25
IOU = 0.45
DEVICE = 0   # set to 'cpu' if needed

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}'
assert VIDEO_PATH.exists(), f'Video not found: {VIDEO_PATH}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Model:', MODEL_PATH)
print('Video:', VIDEO_PATH)
print('Output root:', OUTPUT_ROOT)


Model: /cluster/home/henrban/aquaculture-perception/object-detection/../runs/detect/outputs/training/solaqua_fish/yolov8n_solaqua_fish_120e_fair/weights/best.pt
Video: /cluster/home/henrban/aquaculture-perception/object-detection/../data-processing/vision/SOLAQUA/raw_processed/mp4s/vision_raw_2024-08-20_17-14-36.mp4
Output root: /cluster/home/henrban/aquaculture-perception/object-detection/../runs/video_demos/predictions/solaqua_fish


In [3]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model


/cluster/home/henrban/aquaculture-perception/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

## Run prediction and save annotated video

This uses Ultralytics `predict` mode on the input `.mp4` and writes a saved output video under `OUTPUT_ROOT / RUN_NAME`.


In [4]:
results = model.predict(
    source=str(VIDEO_PATH),
    save=True,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    device=DEVICE,
    stream=False,
    verbose=True,
)

results[:1] if isinstance(results, list) else type(results)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1242) /cluster/home/henrban/aquaculture-perception/object-detection/../data-processing/vision/SOLAQUA/raw_processed/mp4s/vision_raw_2024-08-20_17-14-36.mp4: 384x640 (no detections), 352.7ms
video 1/1 (frame 2/1242) /cluster/home/henrban/aquaculture-perception/object-detection/../data-processing/vision/SOLAQUA/raw_processed/mp4s/vision_raw_2024-08-20_17-14-36.mp4: 384x640 (no detections), 5.1ms
video 1/1 (frame 3/1242) /cluster/home/hen

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'fish'}
 obb: None
 orig_img: array([[[131, 150,  51],
         [140, 159,  60],
         [147, 166,  67],
         ...,
         [148, 170,  55],
         [147, 169,  54],
         [146, 168,  53]],
 
        [[144, 163,  64],
         [147, 166,  67],
         [146, 165,  66],
         ...,
         [150, 172,  57],
         [149, 171,  56],
         [148, 170,  55]],
 
        [[161, 181,  72],
         [156, 176,  67],
         [152, 172,  63],
         ...,
         [152, 174,  59],
         [152, 174,  59],
         [152, 174,  59]],
 
        ...,
 
        [[ 67,  90,  20],
         [ 67,  90,  20],
         [ 68,  91,  21],
         ...,
         [ 66,  83,   7],
         [ 66,  83,   7],
         [ 66,  83,   7]],
 
        [[ 63,  86,  16],
         [ 64,  87,  17],
         [ 64,  87,  17],
         ...,
         [ 66,  83, 

In [5]:
run_dir = OUTPUT_ROOT / RUN_NAME
saved_videos = sorted(list(run_dir.glob('*.mp4')) + list(run_dir.glob('*.avi')) + list(run_dir.glob('*.mov')))

print('Run directory:', run_dir)
print('Saved files:')
for p in sorted(run_dir.iterdir()):
    print(' -', p.name)

if saved_videos:
    output_video = saved_videos[0]
    print('\nAnnotated output video:', output_video)
else:
    print('\nNo saved video file found. Check the run directory above.')


Run directory: /cluster/home/henrban/aquaculture-perception/object-detection/../runs/video_demos/predictions/solaqua_fish/VIDEO_yolov8n_solaqua_fish_120e_fair
Saved files:
 - vision_raw_2024-08-20_17-14-36.avi

Annotated output video: /cluster/home/henrban/aquaculture-perception/object-detection/../runs/video_demos/predictions/solaqua_fish/VIDEO_yolov8n_solaqua_fish_120e_fair/vision_raw_2024-08-20_17-14-36.avi


## Optional: batch over multiple videos in a folder


In [6]:
# Uncomment and edit to run all videos in a directory.
# VIDEO_DIR = Path('/cluster/home/henrban/aquaculture-perception/path/to/videos')
# for video_file in sorted(VIDEO_DIR.glob('*.mp4')):
#     print(f'Running on {video_file.name}')
#     model.predict(
#         source=str(video_file),
#         save=True,
#         project=str(OUTPUT_ROOT),
#         name=video_file.stem,
#         exist_ok=True,
#         imgsz=IMGSZ,
#         conf=CONF,
#         iou=IOU,
#         device=DEVICE,
#         stream=False,
#         verbose=False,
#     )
